# Fase 3 e 4: Validação Cruzada (CV) dos 10 Modelos
**Projeto:** Sistema NDR com ML em Duas Camadas (TCC)

## Objetivos deste Notebook:
1. Carregar os recortes de dados da Camada 1 e Camada 2 gerados anteriormente.
2. Instanciar 10 modelos de Machine Learning utilizando seus hiperparâmetros *default* (com adição apenas de balanceamento de classes).
3. Executar o **Stratified 5-Fold Cross-Validation** para cada modelo, garantindo robustez na avaliação.
4. Coletar e comparar as métricas: `Acurácia`, `Precision`, `Recall`, `F1-Score` e o tempo de treinamento.


In [2]:
# Instalando o XGBoost e LightGBM (caso o Colab precise atualizar)
!pip install xgboost lightgbm -q

import pandas as pd
import numpy as np
import time
import joblib
from google.colab import drive

# Importação dos Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# Importação das Ferramentas de Validação Cruzada
from sklearn.model_selection import cross_validate, StratifiedKFold

# Conectando ao Drive
drive.mount('/content/drive')
pasta_artefatos = '/content/drive/MyDrive/TCC2/artefatos_treino'

print("Carregando arrays da Camada 1...")
X_train_c1 = joblib.load(f'{pasta_artefatos}/X_train_c1.joblib')
y_train_c1 = joblib.load(f'{pasta_artefatos}/y_train_c1.joblib')

print("Carregando arrays da Camada 2...")
X_train_c2 = joblib.load(f'{pasta_artefatos}/X_train_c2.joblib')
y_train_c2 = joblib.load(f'{pasta_artefatos}/y_train_c2.joblib')

print("Dados de treino carregados para a Validação Cruzada!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carregando arrays da Camada 1...
Carregando arrays da Camada 2...
Dados de treino carregados para a Validação Cruzada!


### 3.1 Definição do Dicionário de Modelos
Aqui instanciamos todos os 10 modelos. Para algoritmos que suportam o parâmetro `class_weight='balanced'`, nós o utilizamos como alternativa inteligente ao SMOTE, ensinando o modelo a dar mais atenção às classes com menor quantidade de amostras.


In [3]:
# Dicionário com os 10 algoritmos (Maximizando GPU e CPU)
modelos = {
    '1_Logistic_Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1),
    '2_Decision_Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    '3_Random_Forest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    '4_Extra_Trees': ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    #'5_Gradient_Boosting': GradientBoostingClassifier(random_state=42), # Desativado por lentidão extrema
    '6_XGBoost': XGBClassifier(random_state=42, tree_method='hist', device='cuda'),
    '7_LightGBM': LGBMClassifier(class_weight='balanced', random_state=42, device_type='gpu', n_jobs=-1),
    '8_KNN': KNeighborsClassifier(n_jobs=-1),
    '9_Naive_Bayes': GaussianNB()
    #'10_MLP_Neural_Net': MLPClassifier(random_state=42, max_iter=300) # Opcional: Desative se quiser poupar algumas horas
}

print(f"{len(modelos)} Modelos carregados (XGBoost e LightGBM configurados para GPU)!")


8 Modelos carregados (XGBoost e LightGBM configurados para GPU)!


### 4.1 Loop de Treinamento (Stratified 5-Fold Cross-Validation)
Vamos criar uma função que iterará sobre nossos 10 modelos.
Para cada modelo, ela cortará os dados de Treino em 5 pedaços (Folds). O modelo treinará em 4 e será testado no 5º. Esse processo se repete 5 vezes, e tiramos a **média** das métricas. Isso garante que a nota do modelo não foi "sorte".
As métricas coletadas serão: `Accuracy`, `Precision Macro`, `Recall Macro` e a nossa principal, `F1-Score Macro`.


In [4]:
def executar_cv(modelos_dict, X_train, y_train, nome_camada):
    print(f"Iniciando Validação Cruzada - {nome_camada}")
    print("="*60)

    # 5 Folds estratificados para manter a proporção das classes
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Dicionário com as métricas que queremos que o cross_validate retorne
    scoring = {
        'acc': 'accuracy',
        'prec': 'precision_macro',
        'rec': 'recall_macro',
        'f1': 'f1_macro'
    }

    resultados = []

    for nome, modelo in modelos_dict.items():
        print(f"Treinando e avaliando: {nome}...")
        inicio = time.time()

        # O cross_validate fará todo o trabalho pesado
        # return_train_score=False para poupar memória e tempo
        cv_results = cross_validate(
            modelo, X_train, y_train,
            cv=cv, scoring=scoring, n_jobs=1, # XGBoost GPU não gosta de n_jobs=-1 aqui
            return_train_score=False
        )

        fim = time.time()
        tempo_total = fim - inicio

        # Coletando a MÉDIA (mean) dos 5 Folds para cada métrica
        resultados.append({
            'Modelo': nome,
            'Accuracy': cv_results['test_acc'].mean(),
            'Precision': cv_results['test_prec'].mean(),
            'Recall': cv_results['test_rec'].mean(),
            'F1_Score': cv_results['test_f1'].mean(),
            'Desvio_F1': cv_results['test_f1'].std(), # Quanto o modelo variou entre os Folds
            'Tempo_CV_Segundos': tempo_total
        })

        print(f"   {nome} finalizado em {tempo_total:.2f}s | F1-Score: {cv_results['test_f1'].mean():.4f}")
        print("-" * 40)

    df_resultados = pd.DataFrame(resultados)
    # Ordenar pelos melhores F1-Scores primeiro
    return df_resultados.sort_values(by='F1_Score', ascending=False).reset_index(drop=True)


In [5]:
# Executando a CV para a Camada Binária (BENIGN vs ATTACK)
df_resultados_c1 = executar_cv(modelos, X_train_c1, y_train_c1, "CAMADA 1 (Binária)")

print("\nRANKING FINAL - CAMADA 1:")
display(df_resultados_c1)


Iniciando Validação Cruzada - CAMADA 1 (Binária)
Treinando e avaliando: 1_Logistic_Regression...
   1_Logistic_Regression finalizado em 230.45s | F1-Score: 0.8938
----------------------------------------
Treinando e avaliando: 2_Decision_Tree...
   2_Decision_Tree finalizado em 412.07s | F1-Score: 0.9975
----------------------------------------
Treinando e avaliando: 3_Random_Forest...
   3_Random_Forest finalizado em 353.67s | F1-Score: 0.9976
----------------------------------------
Treinando e avaliando: 4_Extra_Trees...
   4_Extra_Trees finalizado em 247.65s | F1-Score: 0.9971
----------------------------------------
Treinando e avaliando: 6_XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [21:18:50] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


   6_XGBoost finalizado em 15.36s | F1-Score: 0.9985
----------------------------------------
Treinando e avaliando: 7_LightGBM...
[LightGBM] [Info] Number of positive: 272444, number of negative: 1340836
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 9503
[LightGBM] [Info] Number of data points in the train set: 1613280, number of used features: 47
[LightGBM] [Info] Using GPU Device: NVIDIA A100-SXM4-40GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 36 dense feature groups (55.39 MB) transferred to GPU in 0.047264 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 272444, number of negative: 1340836
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 9503
[LightGBM] [Info] Number of data points in the train set: 1613280, number of used features: 47
[LightGBM] [Info] Using GPU Device: NVIDIA A100-SXM4-40GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 36 dense feature groups (55.39 MB) transferred to GPU in 0.041824 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 272444, number of negative: 1340836
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 9493
[LightGBM] [Info] Number of data points in the train set: 1613280, number of used features: 47
[LightGBM] [Info] Using GPU Device: NVIDIA A100-SXM4-40GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 36 dense feature groups (55.39 MB) transferred to GPU in 0.038796 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   7_LightGBM finalizado em 38.21s | F1-Score: 0.9980
----------------------------------------
Treinando e avaliando: 8_KNN...
   8_KNN finalizado em 3521.95s | F1-Score: 0.9896
----------------------------------------
Treinando e avaliando: 9_Naive_Bayes...
   9_Naive_Bayes finalizado em 11.02s | F1-Score: 0.4155
----------------------------------------

RANKING FINAL - CAMADA 1:


,Modelo,Accuracy,Precision,Recall,F1_Score,Desvio_F1,Tempo_CV_Segundos
0,6_XGBoost,0.999172,0.998172,0.998881,0.998526,0.000034,15.363204
1,7_LightGBM,0.998871,0.996831,0.999165,0.997994,0.000063,38.207293
2,3_Random_Forest,0.998653,0.997815,0.997386,0.997600,0.000111,353.668990
3,2_Decision_Tree,0.998597,0.997507,0.997493,0.997500,0.000075,412.067041
4,4_Extra_Trees,0.998373,0.997323,0.996881,0.997102,0.000116,247.654392
5,8_KNN,0.994125,0.988030,0.991124,0.989569,0.000106,3521.951569
6,1_Logistic_Regression,0.932748,0.859218,0.947127,0.893815,0.000692,230.451460
7,9_Naive_Bayes,0.419767,0.608824,0.646994,0.415496,0.004871,11.015608


In [6]:
# Executando a CV para a Camada Multiclasse (Tipos de Ataque)
df_resultados_c2 = executar_cv(modelos, X_train_c2, y_train_c2, "CAMADA 2 (Multiclasse)")

print("\nRANKING FINAL - CAMADA 2:")
display(df_resultados_c2)


Iniciando Validação Cruzada - CAMADA 2 (Multiclasse)
Treinando e avaliando: 1_Logistic_Regression...
   1_Logistic_Regression finalizado em 663.41s | F1-Score: 0.8184
----------------------------------------
Treinando e avaliando: 2_Decision_Tree...
   2_Decision_Tree finalizado em 27.58s | F1-Score: 0.8893
----------------------------------------
Treinando e avaliando: 3_Random_Forest...
   3_Random_Forest finalizado em 40.86s | F1-Score: 0.8866
----------------------------------------
Treinando e avaliando: 4_Extra_Trees...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


   4_Extra_Trees finalizado em 22.24s | F1-Score: 0.8826
----------------------------------------
Treinando e avaliando: 6_XGBoost...
   6_XGBoost finalizado em 10.95s | F1-Score: 0.9068
----------------------------------------
Treinando e avaliando: 7_LightGBM...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 7403
[LightGBM] [Info] Number of data points in the train set: 272444, number of used features: 45
[LightGBM] [Info] Using GPU Device: NVIDIA A100-SXM4-40GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 36 dense feature groups (9.35 MB) transferred to GPU in 0.010917 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 7412
[LightGBM] [Info] Number of data points in the train set: 272444, number of used features: 45
[LightGBM] [Info] Using GPU Device: NVIDIA A100-SXM4-40GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 36 dense feature groups (9.35 MB) transferred to GPU in 0.008751 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from score -2.484907
[LightGBM] [Info] Start training from scor

...

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
3 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", lin

   7_LightGBM finalizado em 38.81s | F1-Score: nan
----------------------------------------
Treinando e avaliando: 8_KNN...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


   8_KNN finalizado em 103.96s | F1-Score: 0.8448
----------------------------------------
Treinando e avaliando: 9_Naive_Bayes...
   9_Naive_Bayes finalizado em 2.65s | F1-Score: 0.6910
----------------------------------------

RANKING FINAL - CAMADA 2:


,Modelo,Accuracy,Precision,Recall,F1_Score,Desvio_F1,Tempo_CV_Segundos
0,6_XGBoost,0.998015,0.916768,0.903811,0.906824,0.022468,10.953339
1,2_Decision_Tree,0.997639,0.893110,0.887269,0.889252,0.009727,27.583285
2,3_Random_Forest,0.997942,0.920347,0.875207,0.886565,0.015920,40.856687
3,4_Extra_Trees,0.997939,0.895662,0.877172,0.882583,0.030525,22.238537
4,8_KNN,0.997504,0.849158,0.843051,0.844766,0.014988,103.955817
5,1_Logistic_Regression,0.990395,0.826733,0.918557,0.818366,0.001731,663.407199
6,9_Naive_Bayes,0.951958,0.678973,0.857846,0.691047,0.002566,2.653418
7,7_LightGBM,NaN,NaN,NaN,NaN,NaN,38.807202


### 4.2 Exportação dos Resultados da CV


In [7]:
caminho_resultados_c1 = '/content/drive/MyDrive/TCC2/resultados_cv_camada1.csv'
caminho_resultados_c2 = '/content/drive/MyDrive/TCC2/resultados_cv_camada2.csv'

# Salvando as tabelas geradas no Drive
df_resultados_c1.to_csv(caminho_resultados_c1, index=False)
df_resultados_c2.to_csv(caminho_resultados_c2, index=False)

print(f"Resultados da Camada 1 salvos em: {caminho_resultados_c1}")
print(f"Resultados da Camada 2 salvos em: {caminho_resultados_c2}")


Resultados da Camada 1 salvos em: /content/drive/MyDrive/TCC2/resultados_cv_camada1.csv
Resultados da Camada 2 salvos em: /content/drive/MyDrive/TCC2/resultados_cv_camada2.csv
